# Hey Monto — Wake Word Training
Run all cells. Downloads `hey_monto.tflite` at the end.

In [ ]:
# Install dependencies
!pip install -q openwakeword==0.6.0
!pip install -q speechbrain datasets torch torchaudio
!pip install -q onnxruntime librosa audiomentations
print('✅ Dependencies installed')

In [ ]:
import os, requests, zipfile, shutil

# Download openWakeWord training scripts from GitHub
!git clone --depth=1 https://github.com/dscripka/openWakeWord /tmp/oww
os.chdir('/tmp/oww')
print('✅ Repo cloned')

In [ ]:
# Generate synthetic training data for 'hey monto'
import subprocess

TARGET_PHRASE = 'hey monto'
OUTPUT_DIR    = '/tmp/hey_monto_model'
N_SAMPLES     = 5000

os.makedirs(OUTPUT_DIR, exist_ok=True)

# Generate TTS samples for the wake phrase
!python3 /tmp/oww/openwakeword/utils/generate_synthetic_data.py \
    --phrase "{TARGET_PHRASE}" \
    --output_dir {OUTPUT_DIR}/positive_samples \
    --n_samples {N_SAMPLES} \
    --tts_engine gtts

print(f'✅ Synthetic samples generated for: {TARGET_PHRASE}')

In [ ]:
# Train the model
!python3 /tmp/oww/openwakeword/train.py \
    --positive_samples_dir {OUTPUT_DIR}/positive_samples \
    --output_dir {OUTPUT_DIR} \
    --phrase "{TARGET_PHRASE}" \
    --n_epochs 30

print('✅ Training complete!')

In [ ]:
# Download the model
from google.colab import files
import glob

tflite_files = glob.glob(f'{OUTPUT_DIR}/**/*.tflite', recursive=True)
onnx_files   = glob.glob(f'{OUTPUT_DIR}/**/*.onnx',   recursive=True)

print(f'Found models: {tflite_files + onnx_files}')

for f in tflite_files + onnx_files:
    files.download(f)
    print(f'⬇️  Downloading: {f}')